In [1]:
!wget https://zhanggroup.org/BioLiP/download/BioLiP_nr.txt.gz -O downloads/biolip.txt.gz
!wget https://zhanggroup.org/BioLiP/data/protein_nr.fasta.gz -O downloads/biolip.fasta.gz

.downloads/biolip.txt.gz: No such file or directory
--2025-09-10 15:06:36--  https://zhanggroup.org/BioLiP/data/protein_nr.fasta.gz
Resolving zhanggroup.org (zhanggroup.org)... 137.132.93.250
Connecting to zhanggroup.org (zhanggroup.org)|137.132.93.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8021454 (7.6M) [application/x-gzip]
Saving to: ‘.downloadsbiolip.fasta.gz’

.downloadsbiolip.fa 100%[===================>]   7.65M   319KB/s    in 24s     

2025-09-10 15:07:01 (321 KB/s) - ‘.downloadsbiolip.fasta.gz’ saved [8021454/8021454]



In [1]:
import pandas as pd
biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)
# reasonable column names for list above
biolip_df.columns = [
    'pdb_id', 'receptor_chain', 'resolution', 'binding_site_number_code',
    'ligand_id', 'ligand_chain', 'ligand_serial_number', 'binding_site_residues_pdb_num',
    'binding_site_residues_renumbered', 'catalytic_site_residues_pdb_num',
    'catalytic_site_residues_renumbered', 'ec_number', 'go_terms',
    'binding_affinity_manual_survey', 'binding_affinity_binding_moad',
    'binding_affinity_pdbbind_cn', 'binding_affinity_bindingdb',
    'uniprot_id', 'pubmed_id', 'ligand_residue_sequence_number',
    'receptor_sequence'
]
biolip_df.dropna(subset=['pdb_id', 'receptor_chain'], inplace=True)
# biolip_df['seq_len'] = biolip_df['receptor_sequence'].apply(len)
biolip_df['fasta_id'] = biolip_df['pdb_id'].str.cat(biolip_df['receptor_chain'].str.strip())

#load biolip fasta file with biopython
from Bio import SeqIO
fasta_sequences = SeqIO.parse(open('downloads/biolip.fasta'), 'fasta')
fasta_dict = {record.id: str(record.seq) for record in fasta_sequences}
biolip_df = biolip_df[biolip_df['fasta_id'].isin(fasta_dict)]
biolip_df['protein_sequence'] = biolip_df['fasta_id'].map(fasta_dict)
biolip_df['seq_len'] = biolip_df['protein_sequence'].apply(len)

/tmp/ipykernel_2855944/838665195.py:2: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  biolip_df = pd.read_csv('downloads/biolip.txt', sep='\t', header=None)


In [2]:
biolip_df['ligand_id'].value_counts().head(20)
#select 200 random rows from each of the top 10 ligands and save to new dataframe
top_20_ligands = biolip_df['ligand_id'].value_counts().head(12).index.tolist()
biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=200, random_state=1) if len(x) >= 200 else x).reset_index(drop=True)
biolip_top_20 = biolip_top_20[biolip_top_20['seq_len'] < 850]

/tmp/ipykernel_2855944/2789014906.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  biolip_top_20 = biolip_df[biolip_df['ligand_id'].isin(top_20_ligands)].groupby('ligand_id').apply(lambda x: x.sample(n=200, random_state=1) if len(x) >= 200 else x).reset_index(drop=True)


In [3]:
def get_binding_side_residues(res_renum_str):
    res = res_renum_str.split(' ')
    res_ind = [int(r[1:]) for r in res]
    return res_ind
def get_binding_site_AA(res_renum_str):
    res = res_renum_str.split(' ')
    res_aa = [r[0] for r in res]
    return res_aa
def get_site_AA(res_ind, seq):
    site_aa = [seq[i-1] for i in res_ind if i-1 < len(seq)]
    return site_aa
biolip_top_20['binding_site_residues_list'] = biolip_top_20['binding_site_residues_renumbered'].apply(get_binding_side_residues)
biolip_top_20['binding_site_AA'] = biolip_top_20['binding_site_residues_renumbered'].apply(get_binding_site_AA)
biolip_top_20['binding_site_AA_seq'] = biolip_top_20.apply(lambda row: get_site_AA(row['binding_site_residues_list'], row['protein_sequence']), axis=1)

In [ ]:
biolip_top_20['go_terms'] = biolip_top_20['go_terms'].fillna('').apply(lambda x: x.split(',') if x != '' else [])
biolip_top_20['go_terms'] = biolip_top_20['go_terms'].apply(lambda x: ['GO:' + term.zfill(7) if not term.startswith('GO:') else term for term in x])
biolip_top_20 = biolip_top_20[biolip_top_20['go_terms'].map(len) > 0]

In [6]:
columns = ['UniprotID', 'AnnotatedIndices', 'GOTerm', 'Sequence']
col_map = {'uniprot_id': 'UniprotID', 
           'binding_site_residues_list': 'AnnotatedIndices', 
           'go_terms': 'GOTerm', 
           'protein_sequence': 'Sequence', 'ligand_id': 'LigandID',}

In [7]:
biolip_top_20 = biolip_top_20[[col for col in col_map.keys()]]
biolip_top_20.rename(columns=col_map, inplace=True)
biolip_top_20.reset_index(drop=True, inplace=True)
biolip_top_20.to_csv('datasets/biolip_dataset.csv', index=False, sep='\t')

In [ ]:
# print(list(fasta_dict.keys())[:20])

In [60]:
print(biolip_df)

      pdb_id receptor_chain  resolution binding_site_number_code ligand_id  \
0       10mh              A        2.55                     BS01       dna   
1       10mh              A        2.55                     BS02       dna   
2       10mh              A        2.55                     BS03       SAH   
3       11as              A        2.50                     BS01       ASN   
4       11ba              A        2.06                     BS01       UPA   
...      ...            ...         ...                      ...       ...   
86453   9vid              B        2.22                     BS01       HEM   
86454   9vmx              D        3.20                     BS01   peptide   
86455   9vmx              D        3.20                     BS02       L9Q   
86456   9vmx              D        3.20                     BS03       L9Q   
86457   9vqm              A        3.50                     BS01        CU   

      ligand_chain  ligand_serial_number  \
0                B 

In [38]:
biolip_top_20.head()

,pdb_id,receptor_chain,resolution,binding_site_number_code,ligand_id,ligand_chain,ligand_serial_number,binding_site_residues_pdb_num,binding_site_residues_renumbered,catalytic_site_residues_pdb_num,...,go_terms,binding_affinity_manual_survey,binding_affinity_binding_moad,binding_affinity_pdbbind_cn,binding_affinity_bindingdb,uniprot_id,pubmed_id,ligand_residue_sequence_number,receptor_sequence,seq_len
0,2i5b,A,2.800,BS01,ADP,A,1,N141 T178 G180 G181 K182 A188 D190 I206 A214 G...,N139 T176 G178 G179 K180 A186 D188 I204 A212 G...,T178 T211 G213 A214 G215 C216,...,"0005524,0005829,0008478,0008902,0008972,000922...",NaN,NaN,NaN,NaN,P39610,16978644.0,301,MHKALTIAGSDSSGGAGIQADLKTFQEKNVYGMTALTVIVAMDPNN...,269
1,2dhr,A,3.900,BS01,ADP,A,1,G199 V200 G201 K202 T203 H204 I334 H338,G57 V58 G59 K60 T61 H62 I192 H196,NaN,...,"0004176,0004222,0005524,0006508,0016020,0016887",NaN,NaN,NaN,NaN,Q5SI82,16762831.0,1001,RARVLTEAPKVTFKDVAGAEEAKEELKEIVEFLKNPSRFHEMGARI...,458
2,8q72,A,4.170,BS01,ADP,A,1,G46 S47 G48 K49 T50 T51 D77 R78 S88 P90 G91 R1070,G46 S47 G48 K49 T50 T51 D77 R78 S88 P90 G91 R725,NaN,...,NaN,NaN,NaN,NaN,NaN,A0A6D0I2P0,38309275.0,1101,MNQVSGLAGKESFILTRIELFNWGGFHGLHQAAIHQDGTAVIGPTG...,750
3,4yj1,A,2.050,BS01,ADP,A,1,P263 L273 A277 V278 G315 F316 H317 G318 K319 S...,P208 L218 A222 V223 G260 F261 H262 G263 K264 S...,NaN,...,"0000166,0000963,0003723,0003729,0005737,000573...",NaN,Kd=885nM,NaN,NaN,Q57ZF2,26117548.0,701,GFSPLMDFFHSVEGRNYGELRSLTNETYQISENVRCTFLSIQSDPF...,600
4,3ruq,A,2.798,BS01,ADP,A,1,L39 G40 P41 G92 T94 T95 K161 G403 G404 V488 E490,L36 G37 P38 G89 T91 T92 K158 G400 G401 V485 E487,D60 T93 T94 D386,...,"0005524,0006457,0016887,0042802,0046872,005108...",NaN,NaN,NaN,NaN,Q877G8,22193720.0,545,QPGVLPENMKRYMGRDAQRMNILAGRIIAETVRSTLGPKGMDKMLV...,516
